# ECI-SLM Kaggle Runbook

Flow:
1. Clone/update repo
2. Install dependencies
3. Run pretrain
4. Run SFT
5. Evaluate pretrain and SFT checkpoints


In [ ]:
%cd /kaggle/working
!rm -rf eci-slm
!git clone https://github.com/rakheOmar/eci-slm.git
%cd /kaggle/working/eci-slm
!git pull --ff-only origin main

In [ ]:
%cd /kaggle/working/eci-slm
!python -m pip install -U pip setuptools wheel
!python -m pip install -e . --no-deps
!python -m pip install datasets sentencepiece tiktoken python-dotenv

In [ ]:
PRETRAIN_STEPS = 12000
BEST_PRETRAIN_STEP = 12000
SFT_STEPS = 2000

print('PRETRAIN_STEPS =', PRETRAIN_STEPS)
print('BEST_PRETRAIN_STEP =', BEST_PRETRAIN_STEP)
print('SFT_STEPS =', SFT_STEPS)

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/eci-slm

PRETRAIN_STEPS=12000

python main.py \
  --mode prepare_and_train \
  --stage pretrain \
  --rebuild_bins \
  --english_ratio 0.93 \
  --data_dir /kaggle/working/artifact \
  --checkpoint_dir /kaggle/working/checkpoints \
  --keep_last_n 5 \
  --strategy auto \
  --precision mixed_fp16 \
  --max_steps "$PRETRAIN_STEPS" \
  --batch_size 16 \
  --grad_accum_steps 2 \
  --learning_rate 2.5e-4 \
  --warmup_steps 1200 \
  --eval_interval 100 \
  --save_interval 100 \
  --log_interval 20

# --mode: run prepare + train in one command
# --stage: choose pretrain pipeline
# --rebuild_bins: regenerate train/val bins before training
# --english_ratio: mix ratio for general English in pretraining data
# --data_dir: output location for pretrain train.bin/val.bin
# --checkpoint_dir: directory for pretrain checkpoints
# --keep_last_n: keep latest N checkpoints plus best
# --strategy: distribution strategy (auto/single/mirrored/cpu)
# --precision: mixed precision policy
# --max_steps: total pretraining steps
# --batch_size: global batch size
# --grad_accum_steps: gradient accumulation microsteps
# --learning_rate: base learning rate for pretraining
# --warmup_steps: LR warmup steps
# --eval_interval: run validation every N steps
# --save_interval: save checkpoint every N steps
# --log_interval: print train logs every N steps

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/eci-slm

BEST_PRETRAIN_STEP=10600
SFT_STEPS=2000

python main.py \
  --mode prepare_and_train \
  --stage sft \
  --rebuild_bins \
  --sft_source_dirs data/instruct \
  --sft_bin_dir /kaggle/working/artifact_sft \
  --init_checkpoint_dir /kaggle/working/checkpoints \
  --init_step "$BEST_PRETRAIN_STEP" \
  --checkpoint_dir /kaggle/working/checkpoints_sft \
  --keep_last_n 5 \
  --strategy auto \
  --precision mixed_fp16 \
  --max_steps "$SFT_STEPS" \
  --batch_size 16 \
  --grad_accum_steps 2 \
  --sft_learning_rate 1e-5 \
  --sft_warmup_steps 200 \
  --eval_interval 50 \
  --save_interval 50 \
  --log_interval 20

# --mode: run prepare + train in one command
# --stage: choose SFT pipeline
# --rebuild_bins: regenerate SFT datasets before training
# --sft_source_dirs: source text directories for SFT examples
# --sft_bin_dir: output location for SFT train/val npz files
# --init_checkpoint_dir: source checkpoint directory for model init
# --init_step: checkpoint step used to initialize model weights
# --checkpoint_dir: directory for SFT checkpoints
# --keep_last_n: keep latest N checkpoints plus best
# --strategy: distribution strategy (auto/single/mirrored/cpu)
# --precision: mixed precision policy
# --max_steps: total SFT steps
# --batch_size: global batch size
# --grad_accum_steps: gradient accumulation microsteps
# --sft_learning_rate: base learning rate for SFT
# --sft_warmup_steps: LR warmup steps for SFT
# --eval_interval: run validation every N steps
# --save_interval: save checkpoint every N steps
# --log_interval: print train logs every N steps

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/eci-slm

BEST_PRETRAIN_STEP=10600

python src/eval.py \
  --checkpoint-dir /kaggle/working/checkpoints \
  --step "$BEST_PRETRAIN_STEP" \
  --repo-root /kaggle/working/eci-slm \
  --tokenizer-path /kaggle/working/eci-slm/artifact/eci_slm_tokenizer.model \
  --output-dir /kaggle/working/results_pretrain \
  --device auto \
  --precision mixed_fp16

# --checkpoint-dir: directory that contains pretrain checkpoints
# --step: pretrain checkpoint step to evaluate
# --repo-root: repo base path used by eval script
# --tokenizer-path: tokenizer model path
# --output-dir: directory for evaluation outputs
# --device: runtime device selection
# --precision: mixed precision policy